<a href="https://colab.research.google.com/github/xbanuelos/BI-Portfolio/blob/main/phase_0/Step_0_Data_Dictionary_Exploratory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Dictionary

**DF customers** — 99,441 rows × 5 columns
- `customer_id` → str, unique ID per order (same customer can appear multiple times)
- `customer_unique_id` → str, true unique customer identifier across orders
- `customer_zip_code_prefix` → int, postal code
- `customer_city` → str, city name
- `customer_state` → str, state abbreviation

**DF geolocation** — 1,000,162 rows × 5 columns
- `geolocation_zip_code_prefix` → int, postal code
- `geolocation_lat` → float
- `geolocation_lng` → float
- `geolocation_city` → str, city name
- `geolocation_state` → str, state abbreviation

**DF order_items** — 112,650 rows × 7 columns
- `order_id` → str, links to orders df
- `order_item_id` → int, item sequence within an order
- `product_id` → str, links to products df
- `seller_id` → str, links to sellers df
- `shipping_limit_date` → timestamp, deadline for seller to hand off to carrier
- `price` → float, product price
- `freight_value` → float, shipping cost paid by the customer

Descriptive stats: avg order price = $120.65 | Q1 = $39.90 |
median = $74.99 | Q3 = $134.90

**DF order_payments** — 103,886 rows × 5 columns
- `order_id` → str, links to orders df
- `payment_sequential` → int, payment sequence for a single order
- `payment_type` → str, payment method (credit card, boleto, etc.)
- `payment_installments` → int, number of installments/mensualidades chosen
- `payment_value` → float, amount paid in that payment entry

**DF reviews** — 99,224 rows × 7 columns
- `review_id` → str, unique review identifier
- `order_id` → str, links to orders df
- `review_score` → int, rating from 1 to 5
- `review_comment_title` → str, short title of the review
- `review_comment_message` → str, full review text
- `review_creation_date` → timestamp, when the customer submitted the review
- `review_answer_timestamp` → timestamp, when Olist/seller responded to the review

**DF orders** — 99,441 rows × 8 columns
- `order_id` → str, primary key across all transactional dfs
- `customer_id` → str, links to customers df
- `order_status` → str, current status of the order
- `order_purchase_timestamp` → timestamp, when customer placed the order
- `order_approved_at` → timestamp, when payment was approved
- `order_delivered_carrier_date` → timestamp, when seller handed off to carrier
- `order_delivered_customer_date` → timestamp, when customer received the order
- `order_estimated_delivery_date` → timestamp, estimated delivery date shown to the customer at purchase

**DF products** — 32,952 rows × 9 columns
- `product_id` → str, links to order_items df
- `product_category_name` → str, category in Portuguese
- `product_name_lenght` → float, character count of the product name
- `product_description_lenght` → float, character count of the description
- `product_photos_qty` → float, number of photos in the listing
- `product_weight_g` → float, weight in grams
- `product_length_cm` → float, length in cm
- `product_height_cm` → float, height in cm
- `product_width_cm` → float, width in cm

**DF sellers** — 3,095 rows × 4 columns
- `seller_id` → str, links to order_items df
- `seller_zip_code_prefix` → int, seller postal code
- `seller_city` → str, seller city
- `seller_state` → str, seller state abbreviation

**DF product_category_name_translation** — 71 rows × 2 columns
- `product_category_name` → str, category name in Portuguese
- `product_category_name_english` → str, category name in English

## Dataset Relationships

The 8 dataframes connect through shared key columns, which allows us to
build a unified view of the full customer journey — from purchase to delivery.

- `orders` is the central table. It connects to almost everything via `order_id`
  and to `customers` via `customer_id`.
- `order_items` links to `orders` via `order_id`, to `products` via `product_id`,
  and to `sellers` via `seller_id`.
- `order_payments` and `reviews` both link to `orders` via `order_id`.
- `products` connects to `order_items` via `product_id`, and its category names
  can be translated using `product_category_name_translation` via
  `product_category_name`.
- `geolocation` is the only standalone table — it connects indirectly via
  `zip_code_prefix`, available in both `customers` and `sellers`.

In practice, most analyses in this portfolio will start from `orders` and
join outward depending on the business question being answered.

In [2]:
import os
import pandas as pd

datasets = {
    'customers' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_customers_dataset.csv'),
    'geolocation' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_geolocation_dataset.csv'),
    'order_items' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_order_items_dataset.csv'),
    'order_payments' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_order_payments_dataset.csv'),
    'reviews' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_order_reviews_dataset.csv'),
    'orders' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_orders_dataset.csv'),
    'products' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_products_dataset.csv'),
    'sellers' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/olist_sellers_dataset.csv'),
    'product_category_name_translation' : pd.read_csv('/content/drive/MyDrive/BI Portfolio/product_category_name_translation.csv')
}

In [6]:
# Quick look at each one
for name, df in datasets.items():
    print(f"\n{name}")
    print(f"  {df.shape[0]:,} rows — columns: {list(df.columns)}")
    display(df.head())
    display(df.describe().round(2))
    nulls = df.isnull().sum()[df.isnull().sum() > 0]
    if len(nulls) > 0:
        print(f"  nulls: {nulls.to_dict()}")



customers
  99,441 rows — columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


,customer_zip_code_prefix
count,99441.00
mean,35137.47
std,29797.94
min,1003.00
25%,11347.00
50%,24416.00
75%,58900.00
max,99990.00



geolocation
  1,000,163 rows — columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


,seller_zip_code_prefix
count,3095.00
mean,32291.06
std,32713.45
min,1001.00
25%,7093.50
50%,14940.00
75%,64552.50
max,99730.00



product_category_name_translation
  71 rows — columns: ['product_category_name', 'product_category_name_english']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1
